In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Find the archive from the notebook folder or either project root.
archive_paths = (
    Path("covariance_forecasts.npz"),
    Path("data/risk/covariance_forecasts.npz"),
    Path("portfolio_modelling/data/risk/covariance_forecasts.npz"),
)
archive_path = next((path for path in archive_paths if path.is_file()), None)

if archive_path is None:
    raise FileNotFoundError(
        f"Could not find covariance_forecasts.npz from {Path.cwd()}"
    )

with np.load(archive_path) as risk:
    preferred_model = risk["preferred_model"].item()
    dates = pd.DatetimeIndex(risk["dates"])
    tickers = risk["tickers"]
    forecasts_by_model = {
        model: risk[model] for model in ("sample", "ledoit_wolf", "ewma")
    }

for model, matrices in forecasts_by_model.items():
    print(f"{model}: {matrices.shape}")  # (dates, assets, assets)

# Change this to "sample", "ledoit_wolf", or "ewma" to view another model.
model_to_view = preferred_model
asset_count = len(tickers)

all_forecasts = pd.DataFrame(
    forecasts_by_model[model_to_view].reshape(len(dates) * asset_count, asset_count),
    index=pd.MultiIndex.from_product([dates, tickers], names=["date", "asset"]),
    columns=pd.Index(tickers, name="asset"),
)

print(f"\nShowing {model_to_view}: {all_forecasts.shape[0]:,} rows × {asset_count} columns")
display(all_forecasts)

In [ ]:
import numpy as np

with np.load("covariance_forecasts.npz") as risk:
    dates = risk["dates"]
    tickers = risk["tickers"]
    daily_covariances = risk[risk["preferred_model"].item()]